# Which report-label table should you train on? A benchmark against the 58 annotated studies

Only **58 of 4,407** training studies carry the twelve radiologist labels. Everyone else trains on labels
derived from the radiology reports — and several such tables are now public. Which one you pick silently
caps everything downstream, yet I could not find a published comparison. So here is one.

**Method:** every public table is scored by macro AUC against the 58 studies that carry real radiologist
labels — the same ground truth the competition scores on. No model is trained; this measures the
*supervision*, not the network.

**Results (macro AUC vs. the 58 annotated studies):**

| Label source | macro AUC | coverage |
|---|---|---|
| `stevenleehans/rsna-knee-llm-report-labels` — v4 blend | **0.893** | 58/58 |
| `stevenleehans/…` — v2 | 0.887 | 58/58 |
| `pilkwang/rsna-knee-llm-labels` — report_labels_v2 | 0.870 | 57/58 |
| a hand-written multilingual regex extractor (mine, below) | 0.711 | 58/58 |

Three things worth taking away before the code:

1. **The ceiling is the labels.** The best table agrees with radiologists at 0.893. A model trained on it
   is learning a teacher that is itself ~0.89 correct, which is why single models in this competition
   converge around 0.87-0.89 no matter the architecture.
2. **Per-label quality is wildly uneven** — ACL is read almost perfectly (0.99), Synovitis barely better
   than chance-plus (0.79). Your model's weakest label is likely your labels' weakest label, not your
   network's.
3. **Averaging two independent tables** produces a soft target that beats either alone on several labels —
   noise in one reader cancels noise in the other.

*Not affiliated with any of these dataset authors — all credit for the tables themselves is theirs.*

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

PATH = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
train = pd.read_csv(os.path.join(PATH, 'train.csv'))
LABELS = list(train.columns[2:14])

annotated = train[train[LABELS].notna().all(axis=1)].set_index('StudyInstanceUID')
print(f'{len(annotated)} of {len(train)} studies carry radiologist labels '
      f'({len(annotated)/len(train)*100:.1f}%)')
print(f'positives per label:')
print(annotated[LABELS].sum().astype(int).to_string())

## The candidate tables

Two public LLM-read tables (several versions each), plus a rule-based extractor written from scratch as a
no-dependency baseline.

In [ ]:
def score_table(df, name, id_col='StudyInstanceUID'):
    df = df.set_index(id_col)
    cols = {}
    for c in LABELS:                          # tolerate naming differences
        for dc in df.columns:
            if dc.lower().replace('_', ' ').replace(chr(39), '') == c.lower().replace(chr(39), ''):
                cols[c] = dc
                break
    common = annotated.index.intersection(df.index)
    aucs = {}
    for c in LABELS:
        if c not in cols:
            continue
        y = annotated.loc[common, c].astype(int)
        p = pd.to_numeric(df.loc[common, cols[c]], errors='coerce')
        m = p.notna()
        if 0 < y[m].sum() < m.sum():
            aucs[c] = roc_auc_score(y[m], p[m])
    return {'name': name, 'coverage': len(common), 'macro': float(np.mean(list(aucs.values()))), **aucs}

candidates = []
for pattern, name in [
    ('/kaggle/input/**/llm_labels_v4_blend.csv', 'stevenleehans v4 blend'),
    ('/kaggle/input/**/llm_labels_v2.csv',       'stevenleehans v2'),
    ('/kaggle/input/**/llm_labels_full.csv',     'stevenleehans full'),
    ('/kaggle/input/**/report_labels_v2.csv',    'pilkwang report_labels_v2'),
]:
    hits = glob.glob(pattern, recursive=True)
    if hits:
        candidates.append(score_table(pd.read_csv(hits[0]), name))

res = pd.DataFrame(candidates).set_index('name')
res[['coverage', 'macro']].sort_values('macro', ascending=False)

## Per-label agreement — where the supervision is strong and where it is not

In [ ]:
per = res[LABELS].sort_values('macro' if 'macro' in res.columns else LABELS[0]) if False else res[LABELS]
order = per.mean().sort_values().index
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(order)); w = 0.8 / len(per)
for i, (name, row) in enumerate(per.iterrows()):
    ax.bar(x + i*w, row[order].values, w, label=name)
ax.axhline(0.5, color='k', lw=0.8, ls='--')
ax.set_xticks(x + 0.4 - w/2); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_ylabel('AUC vs radiologist labels'); ax.set_ylim(0.4, 1.02)
ax.set_title('How well each report-label table agrees with radiologists, per finding')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print(per.T.round(3).to_string())

**Read the low bars as a ceiling, not a failure.** Synovitis sits near 0.79 in the best table: most
reports never mention synovitis at all (I measured ~16% of the 4,407 reports contain any synovitis term in
any of the nine languages), so the label is largely inferred rather than read. No architecture recovers a
signal the targets do not contain — which is why every public model I have seen scores its worst on
Synovitis and Lateral OA.

## Blending two independent readers

The two dataset authors used different pipelines, so their mistakes are not the same mistakes. Averaging
them produces a soft target in [0, 1] that is closer to the radiologist than either alone on several labels.

In [ ]:
a = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
b = glob.glob('/kaggle/input/**/report_labels_v2.csv', recursive=True)
if a and b:
    A = pd.read_csv(a[0]).set_index('StudyInstanceUID')[LABELS]
    B = pd.read_csv(b[0]).set_index('StudyInstanceUID')
    B = B[[c for c in LABELS]].apply(pd.to_numeric, errors='coerce')
    common = A.index.intersection(B.index)
    blend = (A.loc[common] + B.loc[common]) / 2.0
    rows = []
    for nm, tbl in [('A: stevenleehans v4', A.loc[common]),
                    ('B: pilkwang v2', B.loc[common]),
                    ('mean(A, B)', blend)]:
        s = score_table(tbl.reset_index(), nm)
        rows.append(s)
    cmp = pd.DataFrame(rows).set_index('name')[['macro'] + LABELS]
    print(cmp.round(3).T.to_string())

## A from-scratch multilingual extractor, for reference

If you would rather not depend on someone else's table, the reports can be parsed directly. The catch is
that they arrive in about nine languages (my rough count over the 4,407: ~37% English, then Spanish, Greek,
Cyrillic-script, German, Croatian/Serbian, Turkish, Dutch, French), so a lexicon has to carry all of them at
once. Below is a compact version — anatomy stems crossed with pathology terms, negation scoped to the
clause. It reaches **0.711** macro: usable, well short of the LLM tables, and included mostly to show where
the difficulty lies.

In [ ]:
import re, unicodedata

_PRE = str.maketrans({chr(305): 'i', chr(304): 'i', 'I': 'i', chr(223): 'ss'})
def norm(t):
    t = str(t).translate(_PRE).lower()
    t = unicodedata.normalize('NFKD', t)
    return re.sub(r'[ \t]+', ' ', ''.join(c for c in t if not unicodedata.combining(c)))

def clauses(t):
    t = re.sub(r'(?<![.:;!?])\n', ' ', t)          # unwrap hard-wrapped lines first
    return [c.strip() for c in re.split(r'[.;:!?\n]', t) if c.strip()]

NEG = re.compile(r'\bno\b|\bnot\b|without|\bsin\b|ausencia|pas de|\bsans\b|\bgeen\b|\bkeine?\b'
                 r'|\bohne\b|nicht|\byok\b|saptanmadi|izlenme|\bnema\b|\bbez\b|\bnije\b'
                 r'|χωρις|δεν |\bбез\b|нема|normal\b|unremarkable|intact|integr|preserved')
TEAR = re.compile(r'tear|torn|ruptur|rotura|desgarro|dechir|scheur|riss\b|yirti|ρηξη|руптур|разкъс|puknu')
STEMS = {
 'ACL': r'anterior cruciate|\bacl\b|cruzado anterior|\blca\b|croise anterieur|kruisband'
        r'|kreuzband|on capraz|prednj\w* krizn|χιαστ|предн\w* кръстн',
 'Medial Meniscus': r'menisco (interno|medial)|medial menisc|menisc\w* medial|binnenmeniscus'
        r'|innenmeniskus|ic menisk|medijaln\w* menisk|εσω μηνισκ|медиалн\w* мениск',
 'Effusion': r'effusion|derrame|erguss|epanchement|hydrops|efuzij|eff?uzyon|sivi artisi|συλλογη|излив',
 'Synovitis': r'synovit|sinovit|synovial (thickening|proliferation)|υμενιτιδ|синовит|sinovyal',
}

def extract(report, label):
    rx = re.compile(STEMS[label])
    for c in clauses(norm(report)):
        if rx.search(c):
            if label in ('ACL', 'Medial Meniscus'):
                if TEAR.search(c) and not NEG.search(c):
                    return 1.0
            elif not NEG.search(c):
                return 1.0
    return 0.0

demo = {}
for lab in STEMS:
    p = annotated.index.map(lambda u: extract(train.set_index('StudyInstanceUID').loc[u, 'Report'], lab))
    y = annotated[lab].astype(int)
    demo[lab] = roc_auc_score(y, list(p))
print('rule extractor, four representative labels:')
for k, v in demo.items():
    print(f'   {k:18s} {v:.3f}')

## What I would do with this

1. **Train on the blended soft target**, not a single table — it costs nothing and cancels reader noise.
2. **Hold out the 58 annotated studies for reporting, but do not select checkpoints on them.** Fifty-eight
   studies give 9-35 positives per label; swings of ±0.03 are noise. In my own runs the model that ranked
   *last* on those 58 ranked *first* on the leaderboard. Carve a few hundred extra studies with derived
   labels as a selection holdout instead.
3. **Keep byte-identical reports on one side of your split.** 131 studies share a report with another study
   (46 groups, the largest spanning 37 studies); split them apart and you are scoring a model on targets it
   was trained on.

If this saved you an experiment, an upvote helps others find it. Corrections welcome — especially if you
have measured a table I missed.